In [4]:
!pip uninstall -y torchvision torchaudio transformers 2>/dev/null
!pip install -U spacy spacy-transformers
!pip install "transformers==4.46.3" "huggingface-hub==0.26.5"

import torch
print("CUDA:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.2/33.2 MB 63.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.5/780.5 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.4/313.4 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 79.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 113.4 MB/s eta 0:00:0000:010:01
   ━━

In [5]:
!wget https://raw.githubusercontent.com/UniversalDependencies/UD_Northern_Kurdish-Kurmanji/master/kmr_kurmanji-ud-train.conllu
!wget https://raw.githubusercontent.com/UniversalDependencies/UD_Northern_Kurdish-Kurmanji/master/kmr_kurmanji-ud-test.conllu

!wc -l kmr_kurmanji-ud-train.conllu
!wc -l kmr_kurmanji-ud-test.conllu

--2026-05-02 12:38:02--  https://raw.githubusercontent.com/UniversalDependencies/UD_Northern_Kurdish-Kurmanji/master/kmr_kurmanji-ud-train.conllu
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 18331 (18K) [text/plain]
Saving to: ‘kmr_kurmanji-ud-train.conllu’

kmr_kurmanji-ud-tra 100%[===================>]  17.90K  --.-KB/s    in 0s      

2026-05-02 12:38:02 (105 MB/s) - ‘kmr_kurmanji-ud-train.conllu’ saved [18331/18331]

--2026-05-02 12:38:02--  https://raw.githubusercontent.com/UniversalDependencies/UD_Northern_Kurdish-Kurmanji/master/kmr_kurmanji-ud-test.conllu
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.1

In [6]:
import spacy
from spacy.tokens import DocBin

nlp = spacy.blank("xx")

def load_conllu(filepath):
    sentences = []
    tokens, pos_tags, lemmas, deps, heads, morphs = [], [], [], [], [], []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("#"):
                continue
            if not line:
                if tokens:
                    sentences.append((tokens, pos_tags, lemmas, deps, heads, morphs))
                    tokens, pos_tags, lemmas, deps, heads, morphs = [], [], [], [], [], []
                continue
            parts = line.split("\t")
            if len(parts) < 10:
                continue
            if "-" in parts[0] or "." in parts[0]:
                continue

            tokens.append(parts[1])
            lemmas.append(parts[2])
            pos_tags.append(parts[3])
            morphs.append(parts[5])
            heads.append(int(parts[6]))
            deps.append(parts[7])

    if tokens:
        sentences.append((tokens, pos_tags, lemmas, deps, heads, morphs))

    return sentences


def conllu_to_spacy(sentences):
    db = DocBin()
    skipped = 0

    for tokens, pos_tags, lemmas, deps, heads, morphs in sentences:
        words = tokens
        spaces = [True] * len(words)
        doc = spacy.tokens.Doc(nlp.vocab, words=words, spaces=spaces)

        if len(doc) != len(pos_tags):
            skipped += 1
            continue

        for i, token in enumerate(doc):
            token.tag_ = pos_tags[i]
            token.pos_ = pos_tags[i]
            token.lemma_ = lemmas[i]
            if morphs[i] != "_":
                token.set_morph(morphs[i])

        try:
            for i, token in enumerate(doc):
                head_idx = heads[i] - 1
                if head_idx == -1:
                    head_idx = i
                token.head = doc[head_idx]
                token.dep_ = deps[i].lower()
            doc.set_ents([])
            db.add(doc)
        except Exception as e:
            skipped += 1

    print(f"Converted {len(sentences)} sentences, {skipped} skipped")
    return db


import random

# Load the large file (mislabeled as test) as our main dataset
all_sents = load_conllu("kmr_kurmanji-ud-test.conllu")

# Shuffle with fixed seed for reproducibility
random.seed(42)
random.shuffle(all_sents)

# 80/10/10 split
train_end = int(len(all_sents) * 0.80)
dev_end   = int(len(all_sents) * 0.90)

train_sents = all_sents[:train_end]
dev_sents   = all_sents[train_end:dev_end]
test_sents  = all_sents[dev_end:]

conllu_to_spacy(train_sents).to_disk("pos_train.spacy")
conllu_to_spacy(dev_sents).to_disk("pos_dev.spacy")
conllu_to_spacy(test_sents).to_disk("pos_test.spacy")

print(f"Train: {len(train_sents)} sentences")
print(f"Dev:   {len(dev_sents)} sentences")
print(f"Test:  {len(test_sents)} sentences")

Converted 587 sentences, 0 skipped
Converted 73 sentences, 0 skipped
Converted 74 sentences, 0 skipped
Train: 587 sentences
Dev:   73 sentences
Test:  74 sentences


In [7]:
# Check what's actually in the files
!wc -l kmr_kurmanji-ud-train.conllu
!wc -l kmr_kurmanji-ud-test.conllu

# Preview the train file structure
!head -50 kmr_kurmanji-ud-train.conllu

302 kmr_kurmanji-ud-train.conllu
12290 kmr_kurmanji-ud-test.conllu
# sent_id = wiki:wikibank.vislcg.txt:396:10980
# text = Lê dema ku mirov bêje Mîrê Botan, hingî jî, mirov behsa mîrê ku li Cizîra Botan mîr e dike.
1	Lê	lê	CCONJ	cnjcoo	_	5	cc	_	_
2	dema	dema	SCONJ	x	_	5	mark	_	_
3	ku	ku	SCONJ	cnjsub	_	2	fixed	_	_
4	mirov	mirov	NOUN	n	Case=Nom|Definite=Def|Gender=Masc|Number=Sing	5	nsubj	_	_
5	bêje	gotin	VERB	vblex	Mood=Sub|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin	21	advcl	_	_
6	Mîrê	mîr	NOUN	n	Case=Con|Definite=Def|Gender=Masc|Number=Sing	5	obj	_	_
7	Botan	Botan	PROPN	np	Case=Nom|Definite=Def|Gender=Fem|Number=Sing	6	flat	_	SpaceAfter=No
8	,	,	PUNCT	cm	_	5	punct	_	_
9	hingî	hingî	ADV	adv	_	21	advmod	_	_
10	jî	jî	PART	emph	_	21	advmod	_	SpaceAfter=No
11	,	,	PUNCT	cm	_	10	punct	_	_
12	mirov	mirov	NOUN	n	Case=Nom|Definite=Def|Gender=Masc|Number=Sing	21	nsubj	_	_
13	behsa	behs	NOUN	n	Case=Con|Definite=Def|Gender=Fem|Number=Sing	21	obj	_	_
14	mîrê	mîr	NOUN	n	Case=Con|Definite=Def|Gender

In [7]:
config = """
[paths]
train = pos_train.spacy
dev = pos_dev.spacy

[nlp]
lang = "xx"
pipeline = ["transformer", "morphologizer", "tagger", "parser"]
batch_size = 1

[components]

[components.transformer]
factory = "transformer"

[components.transformer.model]
@architectures = "spacy-transformers.TransformerModel.v3"
name = "xlm-roberta-base"
tokenizer_config = {}
transformer_config = {}
mixed_precision = false
grad_scaler_config = {}

[components.transformer.model.get_spans]
@span_getters = "spacy-transformers.strided_spans.v1"
window = 64
stride = 32

[components.morphologizer]
factory = "morphologizer"

[components.morphologizer.model]
@architectures = "spacy.Tagger.v2"
nO = null

[components.morphologizer.model.tok2vec]
@architectures = "spacy-transformers.TransformerListener.v1"
grad_factor = 1.0
upstream = "transformer"
pooling = {"@layers": "reduce_mean.v1"}

[components.tagger]
factory = "tagger"

[components.tagger.model]
@architectures = "spacy.Tagger.v2"
nO = null

[components.tagger.model.tok2vec]
@architectures = "spacy-transformers.TransformerListener.v1"
grad_factor = 1.0
upstream = "transformer"
pooling = {"@layers": "reduce_mean.v1"}

[components.parser]
factory = "parser"

[components.parser.model]
@architectures = "spacy.TransitionBasedParser.v2"
state_type = "parser"
extra_state_tokens = false
hidden_width = 128
maxout_pieces = 3
use_upper = true
nO = null

[components.parser.model.tok2vec]
@architectures = "spacy-transformers.TransformerListener.v1"
grad_factor = 1.0
upstream = "transformer"
pooling = {"@layers": "reduce_mean.v1"}

[training]
seed = 42
gpu_allocator = "pytorch"
dropout = 0.1
max_epochs = 30
patience = 8000
eval_frequency = 200
frozen_components = []
annotating_components = []
dev_corpus = "corpora.dev"
train_corpus = "corpora.train"

[training.logger]
@loggers = "spacy.ConsoleLogger.v1"
progress_bar = true

[training.optimizer]
@optimizers = "Adam.v1"
beta1 = 0.9
beta2 = 0.999
L2_is_weight_decay = true
L2 = 0.01
grad_clip = 0.5
use_averages = false
eps = 0.00000001

[training.optimizer.learn_rate]
@schedules = "warmup_linear.v1"
warmup_steps = 200
total_steps = 20000
initial_rate = 2e-5

[training.batcher]
@batchers = "spacy.batch_by_words.v1"
discard_oversize = true
tolerance = 0.2

[training.batcher.size]
@schedules = "compounding.v1"
start = 50
stop = 150
compound = 1.001

[corpora]

[corpora.train]
@readers = "spacy.Corpus.v1"
path = ${paths.train}
max_length = 64

[corpora.dev]
@readers = "spacy.Corpus.v1"
path = ${paths.dev}
max_length = 64
"""

with open("pos_config.cfg", "w") as f:
    f.write(config)
print("pos_config.cfg written")

pos_config.cfg written


In [8]:
import os, torch, gc

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()

free, total = torch.cuda.mem_get_info()
print(f"Free VRAM: {free/1e9:.2f} GB / {total/1e9:.2f} GB")

!python -m spacy train pos_config.cfg --output pos_output --gpu-id 0

Free VRAM: 15.53 GB / 15.64 GB
✔ Created output directory: pos_output
ℹ Saving to output directory: pos_output
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
tokenizer_config.json: 100%|██████████████████| 25.0/25.0 [00:00<00:00, 155kB/s]
config.json: 100%|█████████████████████████████| 615/615 [00:00<00:00, 7.65MB/s]
sentencepiece.bpe.model: 100%|█████████████| 5.07M/5.07M [00:00<00:00, 23.8MB/s]
tokenizer.json: 9.10MB [00:00, 31.6MB/s]
2026-05-02 12:38:25.548171: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777725505.753260     163 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777725505.820616     163 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for

train cell

In [9]:
# Evaluate model-last instead
!python -m spacy evaluate pos_output/model-last pos_dev.spacy --gpu-id 0

ℹ Using GPU: 0
2026-05-02 12:53:52.834457: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777726432.856702     196 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777726432.863764     196 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777726432.882779     196 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777726432.882810     196 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777726432.882814     196 computation_placer.cc:177] computa

The evaluate cell 

In [12]:
import spacy

nlp = spacy.load("pos_output/model-last")

# Use proper Kurmanji text with diacritics
texts = [
    "Ev kitêb baş e",
    "Li Kurdistanê gelek mirov dijîn",
    "Mîrê Botan li Cizîrê dijî",
]

for text in texts:
    doc = nlp(text)
    print(f"\nText: {text}")
    for token in doc:
        print(f"  {token.text:15} POS: {token.pos_:8} Lemma: {token.lemma_:15} Dep: {token.dep_}")

ValueError: [E002] Can't find factory for 'transformer' for language MultiLanguage (xx). This usually happens when spaCy calls `nlp.create_pipe` with a custom component name that's not registered on the current language class. If you're using a custom component, make sure you've added the decorator `@Language.component` (for function components) or `@Language.factory` (for class components).

Available factories: merge_noun_chunks, merge_entities, merge_subtokens, ja.morphologizer, attribute_ruler, entity_linker, entity_ruler, lemmatizer, textcat, token_splitter, doc_cleaner, tok2vec, senter, morphologizer, spancat, spancat_singlelabel, future_entity_ruler, span_ruler, trainable_lemmatizer, textcat_multilabel, span_finder, ner, beam_ner, parser, beam_parser, tagger, nn_labeller, sentencizer

test

In [13]:
import os

# Check what was actually saved
print("model-best exists:", os.path.exists("pos_output/model-best"))
print("model-last exists:", os.path.exists("pos_output/model-last"))

# List contents
if os.path.exists("pos_output/model-best"):
    print("\nmodel-best contents:")
    for f in os.listdir("pos_output/model-best"):
        print(" ", f)

if os.path.exists("pos_output/model-last"):
    print("\nmodel-last contents:")
    for f in os.listdir("pos_output/model-last"):
        print(" ", f)

model-best exists: True
model-last exists: True

model-best contents:
  vocab
  parser
  meta.json
  transformer
  tokenizer
  morphologizer
  config.cfg
  tagger

model-last contents:
  vocab
  parser
  meta.json
  transformer
  tokenizer
  morphologizer
  config.cfg
  tagger


In [14]:
import shutil

shutil.make_archive("/kaggle/working/kurdish-pos-output", "zip", "pos_output")
print("Done - download kurdish-pos-output.zip from the output panel")

Done - download kurdish-pos-output.zip from the output panel
